# Course 6: Geometric Deep Learning and Graphs
## Session 1: The geometric blueprint

**Goal:** show with numbers that sum pooling is permutation-invariant and a GCN layer is permutation-equivariant.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. A small graph and a random relabelling
A permutation matrix `P` relabels the nodes: features become `P X` and the adjacency becomes `P A Pᵀ`.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
n, d = 6, 4
A = np.triu(rng.random((n, n)) < 0.5, 1); A = (A | A.T).astype(float)
X = rng.normal(size=(n, d))
perm = rng.permutation(n)
P = np.eye(n)[perm]
print("permutation:", perm)

### 2. Sum pooling is invariant
Invariant means: relabel the input and the output does not change.

In [ ]:
pool = lambda X: X.sum(axis=0)
print("pool(X)   =", pool(X).round(4))
print("pool(P X) =", pool(P @ X).round(4))
assert np.allclose(pool(X), pool(P @ X))

### 3. A GCN layer is equivariant
Equivariant means: relabel the input and the output is relabelled the same way, `f(PX, PAPᵀ) = P f(X, A)`.

In [ ]:
def gcn_layer(X, A, W):
    A_hat = A + np.eye(len(A))
    d = A_hat.sum(1)
    A_hat = A_hat / np.sqrt(np.outer(d, d))
    return np.maximum(A_hat @ X @ W, 0)

W = rng.normal(size=(d, 3))
lhs = gcn_layer(P @ X, P @ A @ P.T, W)
rhs = P @ gcn_layer(X, A, W)
print("largest difference:", np.abs(lhs - rhs).max())
assert np.allclose(lhs, rhs)

### 4. Equivariant layers followed by invariant pooling give an invariant graph model

In [ ]:
graph_embed = lambda X, A: pool(gcn_layer(gcn_layer(X, A, W), A, rng_W2))
rng_W2 = rng.normal(size=(3, 3))
assert np.allclose(graph_embed(X, A), graph_embed(P @ X, P @ A @ P.T))
print("graph embedding is the same for both labellings")

### 5. A layer that breaks the symmetry
Flattening the node features and using an ordinary dense layer is not equivariant. This is why we do not feed graphs to an MLP.

In [ ]:
W_flat = rng.normal(size=(n * d, 3))
mlp = lambda X: X.reshape(-1) @ W_flat
print("MLP(X)   =", mlp(X).round(3))
print("MLP(P X) =", mlp(P @ X).round(3), "  <- different")
assert not np.allclose(mlp(X), mlp(P @ X))

### TODO
1. Replace sum pooling with mean and max. Are they invariant? Check.
2. Is concatenating the first two rows of `X` invariant? Predict, then check.
3. Show that a 1-D convolution is equivariant to circular shifts (`np.roll`).